# CloudConditionedLSTM Evaluation

Deep testing of the CloudConditionedLSTMForecaster:
1. **Data Loading**: Use `wrdata` to get real market data
2. **Self-Comparison Tests**: Compare model with different configurations
3. **Model Comparison**: Compare against other FracTime forecasters
4. **Visualization**: Interactive forecast evaluation with `wrchart`

In [ ]:
# Standard imports
import numpy as np
import polars as pl
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Data
from wrdata import DataStream

# FracTime
import fractime as ft
from fractime.baselines import (
    CloudConditionedLSTMForecaster,
    FractalLSTMForecaster,
    ARIMAForecaster,
    LSTMForecaster,
)

# Visualization
try:
    import wrchart as wrc
    WRCHART_AVAILABLE = True
except ImportError:
    WRCHART_AVAILABLE = False
    print("wrchart not available, using matplotlib fallback")
    import matplotlib.pyplot as plt

print(f"FracTime version: {ft.__version__}")
print(f"wrchart available: {WRCHART_AVAILABLE}")

## 1. Data Loading

Load multiple assets for testing - we want variety in market behavior.

In [ ]:
# Initialize data stream
stream = DataStream()

# Define test assets
ASSETS = {
    'SPY': 'US Equity (trending)',
    'GLD': 'Gold ETF (commodity)',
    'TLT': 'Long-term Treasury (bonds)',
    'BTC-USD': 'Bitcoin (crypto)',
}

# Date range
END_DATE = datetime.now().strftime('%Y-%m-%d')
START_DATE = (datetime.now() - timedelta(days=365*3)).strftime('%Y-%m-%d')  # 3 years

print(f"Fetching data from {START_DATE} to {END_DATE}")

In [ ]:
# Fetch data for all assets
data = {}

for symbol, description in ASSETS.items():
    try:
        df = stream.get(symbol, start=START_DATE, end=END_DATE, interval='1d')
        data[symbol] = df
        print(f"✓ {symbol}: {len(df)} days - {description}")
    except Exception as e:
        print(f"✗ {symbol}: Failed - {e}")

print(f"\nLoaded {len(data)} assets")

In [ ]:
# Quick look at the data
for symbol, df in data.items():
    prices = df['close'].to_numpy()
    returns = np.diff(np.log(prices))
    
    # Basic fractal analysis
    analyzer = ft.Analyzer(prices)
    
    print(f"\n{symbol}:")
    print(f"  Price range: ${prices.min():.2f} - ${prices.max():.2f}")
    print(f"  Annualized volatility: {np.std(returns) * np.sqrt(252):.1%}")
    print(f"  Hurst exponent: {analyzer.hurst.value:.3f}")
    print(f"  Regime: {analyzer.result.regime}")

## 2. Evaluation Metrics

Define metrics for comparing forecasts.

In [ ]:
def compute_forecast_metrics(actual: np.ndarray, forecast_result, model_name: str) -> dict:
    """
    Compute evaluation metrics for a forecast.
    
    Args:
        actual: Actual prices during forecast period
        forecast_result: ForecastResult or dict with forecast
        model_name: Name for identification
    
    Returns:
        Dictionary of metrics
    """
    # Get forecast values
    if hasattr(forecast_result, 'forecast'):
        forecast = forecast_result.forecast
        mean = forecast_result.mean
        lower = forecast_result.lower
        upper = forecast_result.upper
    else:
        forecast = forecast_result.get('forecast', forecast_result.get('mean'))
        mean = forecast_result.get('mean', forecast)
        lower = forecast_result.get('lower', forecast * 0.95)
        upper = forecast_result.get('upper', forecast * 1.05)
    
    # Ensure same length
    n = min(len(actual), len(forecast))
    actual = actual[:n]
    forecast = forecast[:n]
    mean = mean[:n]
    lower = lower[:n]
    upper = upper[:n]
    
    # Point forecast metrics
    mae = np.mean(np.abs(actual - forecast))
    rmse = np.sqrt(np.mean((actual - forecast) ** 2))
    mape = np.mean(np.abs((actual - forecast) / actual)) * 100
    
    # Direction accuracy
    actual_direction = np.sign(np.diff(actual))
    forecast_direction = np.sign(np.diff(forecast))
    direction_accuracy = np.mean(actual_direction == forecast_direction) * 100
    
    # Coverage (what % of actuals fall within CI)
    coverage = np.mean((actual >= lower) & (actual <= upper)) * 100
    
    # CI width (average)
    ci_width = np.mean(upper - lower)
    ci_width_pct = np.mean((upper - lower) / forecast) * 100
    
    # Bias
    bias = np.mean(forecast - actual)
    bias_pct = np.mean((forecast - actual) / actual) * 100
    
    return {
        'model': model_name,
        'mae': mae,
        'rmse': rmse,
        'mape': mape,
        'direction_accuracy': direction_accuracy,
        'coverage': coverage,
        'ci_width': ci_width,
        'ci_width_pct': ci_width_pct,
        'bias': bias,
        'bias_pct': bias_pct,
    }


def display_metrics(metrics_list: list):
    """Display metrics as a formatted table."""
    df = pl.DataFrame(metrics_list)
    
    # Format for display
    print("\n" + "=" * 100)
    print("FORECAST EVALUATION METRICS")
    print("=" * 100)
    print(f"{'Model':<30} {'MAE':>10} {'RMSE':>10} {'MAPE%':>10} {'Dir Acc%':>10} {'Coverage%':>10} {'CI Width%':>10}")
    print("-" * 100)
    
    for m in metrics_list:
        print(f"{m['model']:<30} {m['mae']:>10.2f} {m['rmse']:>10.2f} {m['mape']:>10.1f} {m['direction_accuracy']:>10.1f} {m['coverage']:>10.1f} {m['ci_width_pct']:>10.1f}")
    
    print("=" * 100)
    return df

## 3. Self-Comparison Tests

Test CloudConditionedLSTM with different configurations.

In [ ]:
# Use SPY for self-comparison (most liquid, well-behaved)
test_symbol = 'SPY'
df = data[test_symbol]

# Split into train and test
FORECAST_STEPS = 20  # 20 trading days (~1 month)
train_df = df.head(len(df) - FORECAST_STEPS)
test_df = df.tail(FORECAST_STEPS)

train_prices = train_df['close'].to_numpy()
test_prices = test_df['close'].to_numpy()

print(f"Training on {len(train_prices)} days")
print(f"Testing on {len(test_prices)} days")
print(f"Last training price: ${train_prices[-1]:.2f}")
print(f"Test period: ${test_prices[0]:.2f} -> ${test_prices[-1]:.2f}")

In [ ]:
# Define configurations to test
CLOUD_LSTM_CONFIGS = [
    {
        'name': 'CloudLSTM-Default',
        'params': {
            'lookback': 30,
            'n_cloud_paths': 500,
            'cloud_weight': 0.3,
            'hidden_size_1': 64,
            'hidden_size_2': 32,
            'epochs': 50,
            'verbose': 0,
        }
    },
    {
        'name': 'CloudLSTM-HighCloud',
        'params': {
            'lookback': 30,
            'n_cloud_paths': 500,
            'cloud_weight': 0.6,  # Higher cloud weight
            'hidden_size_1': 64,
            'hidden_size_2': 32,
            'epochs': 50,
            'verbose': 0,
        }
    },
    {
        'name': 'CloudLSTM-LowCloud',
        'params': {
            'lookback': 30,
            'n_cloud_paths': 500,
            'cloud_weight': 0.1,  # Lower cloud weight (more LSTM)
            'hidden_size_1': 64,
            'hidden_size_2': 32,
            'epochs': 50,
            'verbose': 0,
        }
    },
    {
        'name': 'CloudLSTM-LargeLSTM',
        'params': {
            'lookback': 30,
            'n_cloud_paths': 500,
            'cloud_weight': 0.3,
            'hidden_size_1': 128,  # Larger LSTM
            'hidden_size_2': 64,
            'epochs': 50,
            'verbose': 0,
        }
    },
    {
        'name': 'CloudLSTM-LongLookback',
        'params': {
            'lookback': 60,  # Longer lookback
            'n_cloud_paths': 500,
            'cloud_weight': 0.3,
            'hidden_size_1': 64,
            'hidden_size_2': 32,
            'epochs': 50,
            'verbose': 0,
        }
    },
    {
        'name': 'CloudLSTM-MorePaths',
        'params': {
            'lookback': 30,
            'n_cloud_paths': 1000,  # More cloud paths
            'cloud_weight': 0.3,
            'hidden_size_1': 64,
            'hidden_size_2': 32,
            'epochs': 50,
            'verbose': 0,
        }
    },
]

print(f"Testing {len(CLOUD_LSTM_CONFIGS)} configurations")

In [ ]:
# Run self-comparison tests
self_comparison_results = []
self_comparison_forecasts = {}

for config in CLOUD_LSTM_CONFIGS:
    name = config['name']
    params = config['params']
    
    print(f"\nTraining {name}...")
    
    try:
        # Create and train model
        model = CloudConditionedLSTMForecaster(**params)
        model.fit(train_prices)
        
        # Generate forecast
        result = model.predict(n_steps=FORECAST_STEPS, n_simulations=100)
        self_comparison_forecasts[name] = result
        
        # Compute metrics
        metrics = compute_forecast_metrics(test_prices, result, name)
        metrics['epochs_trained'] = model.get_model_params().get('epochs_trained', 0)
        self_comparison_results.append(metrics)
        
        print(f"  ✓ MAPE: {metrics['mape']:.1f}%, Dir Acc: {metrics['direction_accuracy']:.1f}%")
        
    except Exception as e:
        print(f"  ✗ Failed: {e}")

print("\n" + "="*60)
print("Self-Comparison Complete")

In [ ]:
# Display self-comparison results
self_comparison_df = display_metrics(self_comparison_results)

## 4. Model Comparison

Compare CloudConditionedLSTM against other forecasting models.

In [ ]:
# Define competitor models
COMPETITOR_MODELS = [
    {
        'name': 'FractalLSTM',
        'class': FractalLSTMForecaster,
        'params': {
            'lookback': 30,
            'hidden_size_1': 64,
            'hidden_size_2': 32,
            'epochs': 50,
            'verbose': 0,
        }
    },
    {
        'name': 'VanillaLSTM',
        'class': LSTMForecaster,
        'params': {
            'lookback': 30,
            'units': 64,
            'epochs': 50,
            'verbose': 0,
        }
    },
    {
        'name': 'ARIMA',
        'class': ARIMAForecaster,
        'params': {}
    },
    {
        'name': 'FractalSimulator',
        'class': ft.Forecaster,
        'params': {}
    },
]

print(f"Comparing against {len(COMPETITOR_MODELS)} models")

In [ ]:
# Run model comparison
comparison_results = []
comparison_forecasts = {}

# Add best CloudLSTM config
best_config = CLOUD_LSTM_CONFIGS[0]  # Default
comparison_forecasts['CloudConditionedLSTM'] = self_comparison_forecasts[best_config['name']]
comparison_results.append(next(r for r in self_comparison_results if r['model'] == best_config['name']))
comparison_results[-1]['model'] = 'CloudConditionedLSTM'

for model_config in COMPETITOR_MODELS:
    name = model_config['name']
    model_class = model_config['class']
    params = model_config['params']
    
    print(f"\nTraining {name}...")
    
    try:
        # Create and train model
        model = model_class(**params)
        model.fit(train_prices)
        
        # Generate forecast
        if name == 'FractalSimulator':
            result = model.predict(steps=FORECAST_STEPS)
        else:
            result = model.predict(n_steps=FORECAST_STEPS)
        
        comparison_forecasts[name] = result
        
        # Compute metrics
        metrics = compute_forecast_metrics(test_prices, result, name)
        comparison_results.append(metrics)
        
        print(f"  ✓ MAPE: {metrics['mape']:.1f}%, Dir Acc: {metrics['direction_accuracy']:.1f}%")
        
    except Exception as e:
        print(f"  ✗ Failed: {e}")

print("\n" + "="*60)
print("Model Comparison Complete")

In [ ]:
# Display comparison results
comparison_df = display_metrics(comparison_results)

## 5. Multi-Asset Test

Test the best CloudLSTM configuration on all assets.

In [ ]:
# Run on all assets
multi_asset_results = []
multi_asset_forecasts = {}

for symbol, df in data.items():
    print(f"\nTesting on {symbol}...")
    
    # Split data
    train_df = df.head(len(df) - FORECAST_STEPS)
    test_df = df.tail(FORECAST_STEPS)
    train_prices = train_df['close'].to_numpy()
    test_prices = test_df['close'].to_numpy()
    
    # Train CloudConditionedLSTM
    try:
        model = CloudConditionedLSTMForecaster(
            lookback=30,
            n_cloud_paths=500,
            cloud_weight=0.3,
            epochs=50,
            verbose=0,
        )
        model.fit(train_prices)
        result = model.predict(n_steps=FORECAST_STEPS, n_simulations=100)
        
        multi_asset_forecasts[symbol] = {
            'train_prices': train_prices,
            'test_prices': test_prices,
            'result': result,
        }
        
        metrics = compute_forecast_metrics(test_prices, result, symbol)
        multi_asset_results.append(metrics)
        
        print(f"  ✓ MAPE: {metrics['mape']:.1f}%, Dir Acc: {metrics['direction_accuracy']:.1f}%")
        
    except Exception as e:
        print(f"  ✗ Failed: {e}")

print("\n" + "="*60)
print("Multi-Asset Test Complete")

In [ ]:
# Display multi-asset results
multi_asset_df = display_metrics(multi_asset_results)

## 6. Visualization

Interactive visualization of forecasts using wrchart.

In [ ]:
def plot_forecast_comparison(train_prices, test_prices, forecasts_dict, title="Forecast Comparison"):
    """
    Plot multiple forecasts on the same chart for comparison.
    """
    if WRCHART_AVAILABLE:
        # Use wrchart
        chart = wrc.Chart(width=1200, height=700, title=title)
        
        # Historical prices
        n_history = min(100, len(train_prices))  # Show last 100 days of history
        history = train_prices[-n_history:]
        x_history = list(range(n_history))
        
        chart.add_line(
            pl.DataFrame({'x': x_history, 'y': history.tolist()}),
            time_col='x',
            value_col='y',
            name='Historical',
            color='#888888',
            width=2,
        )
        
        # Actual test prices
        x_test = list(range(n_history, n_history + len(test_prices)))
        chart.add_line(
            pl.DataFrame({'x': x_test, 'y': test_prices.tolist()}),
            time_col='x',
            value_col='y',
            name='Actual',
            color='#FFFFFF',
            width=3,
        )
        
        # Forecasts
        colors = ['#00FF00', '#FF6B6B', '#4ECDC4', '#FFE66D', '#95E1D3']
        
        for i, (name, result) in enumerate(forecasts_dict.items()):
            if hasattr(result, 'forecast'):
                forecast = result.forecast
            else:
                forecast = result.get('forecast', result.get('mean'))
            
            forecast = forecast[:len(test_prices)]
            x_forecast = list(range(n_history, n_history + len(forecast)))
            
            chart.add_line(
                pl.DataFrame({'x': x_forecast, 'y': forecast.tolist()}),
                time_col='x',
                value_col='y',
                name=name,
                color=colors[i % len(colors)],
                width=2,
            )
        
        chart.show()
        return chart
        
    else:
        # Matplotlib fallback
        fig, ax = plt.subplots(figsize=(14, 8))
        
        n_history = min(100, len(train_prices))
        history = train_prices[-n_history:]
        x_history = np.arange(n_history)
        x_test = np.arange(n_history, n_history + len(test_prices))
        
        ax.plot(x_history, history, color='gray', linewidth=2, label='Historical')
        ax.plot(x_test, test_prices, color='black', linewidth=3, label='Actual')
        
        colors = ['green', 'red', 'blue', 'orange', 'purple']
        
        for i, (name, result) in enumerate(forecasts_dict.items()):
            if hasattr(result, 'forecast'):
                forecast = result.forecast
                lower = result.lower
                upper = result.upper
            else:
                forecast = result.get('forecast', result.get('mean'))
                lower = result.get('lower', forecast * 0.95)
                upper = result.get('upper', forecast * 1.05)
            
            forecast = forecast[:len(test_prices)]
            lower = lower[:len(test_prices)]
            upper = upper[:len(test_prices)]
            x_forecast = np.arange(n_history, n_history + len(forecast))
            
            color = colors[i % len(colors)]
            ax.plot(x_forecast, forecast, color=color, linewidth=2, label=name)
            ax.fill_between(x_forecast, lower, upper, color=color, alpha=0.2)
        
        ax.axvline(x=n_history, color='gray', linestyle='--', alpha=0.5)
        ax.legend(loc='upper left')
        ax.set_title(title)
        ax.set_xlabel('Days')
        ax.set_ylabel('Price')
        ax.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        return fig

In [ ]:
# Plot model comparison for SPY
df = data['SPY']
train_prices = df['close'].to_numpy()[:-FORECAST_STEPS]
test_prices = df['close'].to_numpy()[-FORECAST_STEPS:]

plot_forecast_comparison(
    train_prices,
    test_prices,
    comparison_forecasts,
    title="SPY Forecast Comparison: All Models"
)

In [ ]:
# Plot self-comparison for SPY
plot_forecast_comparison(
    train_prices,
    test_prices,
    self_comparison_forecasts,
    title="SPY Forecast: CloudLSTM Configuration Comparison"
)

In [ ]:
def plot_single_forecast_detail(train_prices, test_prices, result, model_name, title=None):
    """
    Detailed plot of a single forecast with confidence intervals and paths.
    """
    if WRCHART_AVAILABLE:
        try:
            # Try to use ForecastChart
            from wrchart import ForecastChart
            
            # Build result dict
            if hasattr(result, 'paths'):
                result_dict = {
                    'paths': result.paths,
                    'probabilities': result.probabilities,
                    'weighted_forecast': result.forecast,
                }
            else:
                result_dict = result
            
            chart = ForecastChart(
                width=1200,
                height=700,
                title=title or f"{model_name} Forecast",
            )
            chart.set_data(train_prices[-100:], result_dict)
            chart.colorscale('viridis')
            chart.show_percentiles(True)
            chart.show_weighted_forecast(True)
            chart.show()
            return chart
            
        except ImportError:
            pass  # Fall through to line chart
    
    # Fallback to basic visualization
    fig, axes = plt.subplots(2, 1, figsize=(14, 10))
    
    # Top: Forecast with CI
    ax = axes[0]
    n_history = min(100, len(train_prices))
    history = train_prices[-n_history:]
    x_history = np.arange(n_history)
    x_test = np.arange(n_history, n_history + len(test_prices))
    
    ax.plot(x_history, history, color='gray', linewidth=2, label='Historical')
    ax.plot(x_test, test_prices, color='black', linewidth=3, label='Actual')
    
    if hasattr(result, 'forecast'):
        forecast = result.forecast[:len(test_prices)]
        lower = result.lower[:len(test_prices)]
        upper = result.upper[:len(test_prices)]
        
        # 95% CI
        ax.fill_between(x_test, lower, upper, color='green', alpha=0.2, label='95% CI')
        
        # 50% CI
        q25 = result.quantile(0.25)[:len(test_prices)]
        q75 = result.quantile(0.75)[:len(test_prices)]
        ax.fill_between(x_test, q25, q75, color='green', alpha=0.4, label='50% CI')
        
        ax.plot(x_test, forecast, color='green', linewidth=2, label='Forecast')
    else:
        forecast = result.get('forecast', result.get('mean'))[:len(test_prices)]
        ax.plot(x_test, forecast, color='green', linewidth=2, label='Forecast')
    
    ax.axvline(x=n_history, color='gray', linestyle='--', alpha=0.5)
    ax.legend(loc='upper left')
    ax.set_title(title or f"{model_name} Forecast")
    ax.set_ylabel('Price')
    ax.grid(True, alpha=0.3)
    
    # Bottom: Path density (if available)
    ax = axes[1]
    if hasattr(result, 'paths'):
        paths = result.paths[:, :len(test_prices)]
        
        # Plot sample of paths
        n_sample = min(50, len(paths))
        for i in range(n_sample):
            ax.plot(x_test, paths[i], color='green', alpha=0.1, linewidth=0.5)
        
        ax.plot(x_test, test_prices, color='black', linewidth=2, label='Actual')
        ax.set_title('Monte Carlo Paths')
    else:
        ax.text(0.5, 0.5, 'No path data available', ha='center', va='center', transform=ax.transAxes)
    
    ax.set_xlabel('Days')
    ax.set_ylabel('Price')
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    return fig

In [ ]:
# Detailed plot for CloudConditionedLSTM on SPY
plot_single_forecast_detail(
    train_prices,
    test_prices,
    comparison_forecasts['CloudConditionedLSTM'],
    'CloudConditionedLSTM',
    title='CloudConditionedLSTM Forecast on SPY (Detailed)'
)

In [ ]:
# Plot each asset
for symbol, forecast_data in multi_asset_forecasts.items():
    print(f"\n{'='*60}")
    print(f"Asset: {symbol}")
    print(f"{'='*60}")
    
    plot_single_forecast_detail(
        forecast_data['train_prices'],
        forecast_data['test_prices'],
        forecast_data['result'],
        'CloudConditionedLSTM',
        title=f'CloudConditionedLSTM Forecast on {symbol}'
    )

## 7. Sanity Checks

Verify the forecasts are reasonable.

In [ ]:
def run_sanity_checks(result, train_prices, test_prices, model_name):
    """
    Run sanity checks on forecast.
    """
    print(f"\nSanity Checks for {model_name}:")
    print("-" * 40)
    
    if hasattr(result, 'forecast'):
        forecast = result.forecast
        lower = result.lower
        upper = result.upper
        paths = result.paths if hasattr(result, 'paths') else None
    else:
        forecast = result.get('forecast', result.get('mean'))
        lower = result.get('lower', forecast * 0.95)
        upper = result.get('upper', forecast * 1.05)
        paths = None
    
    last_price = train_prices[-1]
    
    checks_passed = 0
    total_checks = 0
    
    # 1. Forecast starts near last price
    total_checks += 1
    start_deviation = abs(forecast[0] - last_price) / last_price * 100
    if start_deviation < 5:  # Within 5%
        print(f"✓ Forecast starts near last price (deviation: {start_deviation:.1f}%)")
        checks_passed += 1
    else:
        print(f"✗ Forecast starts far from last price (deviation: {start_deviation:.1f}%)")
    
    # 2. Forecast is positive (for prices)
    total_checks += 1
    if np.all(forecast > 0) and np.all(lower > 0):
        print(f"✓ All forecast values are positive")
        checks_passed += 1
    else:
        print(f"✗ Some forecast values are non-positive")
    
    # 3. CI makes sense (lower < forecast < upper)
    total_checks += 1
    if np.all(lower <= forecast) and np.all(forecast <= upper):
        print(f"✓ Confidence intervals are properly ordered")
        checks_passed += 1
    else:
        print(f"✗ Confidence intervals are not properly ordered")
    
    # 4. CI width is reasonable (not too narrow, not too wide)
    total_checks += 1
    ci_width_pct = np.mean((upper - lower) / forecast) * 100
    if 1 < ci_width_pct < 50:  # Between 1% and 50%
        print(f"✓ CI width is reasonable ({ci_width_pct:.1f}%)")
        checks_passed += 1
    else:
        print(f"✗ CI width is suspicious ({ci_width_pct:.1f}%)")
    
    # 5. Forecast doesn't explode
    total_checks += 1
    max_change = abs(forecast[-1] - last_price) / last_price * 100
    if max_change < 50:  # Less than 50% change over forecast horizon
        print(f"✓ Forecast doesn't explode (max change: {max_change:.1f}%)")
        checks_passed += 1
    else:
        print(f"✗ Forecast may be exploding (max change: {max_change:.1f}%)")
    
    # 6. Paths are diverse (if available)
    if paths is not None:
        total_checks += 1
        path_std = np.std(paths[:, -1])
        path_diversity = path_std / np.mean(paths[:, -1]) * 100
        if 0.5 < path_diversity < 30:  # Between 0.5% and 30%
            print(f"✓ Monte Carlo paths show healthy diversity ({path_diversity:.1f}%)")
            checks_passed += 1
        else:
            print(f"✗ Monte Carlo paths diversity is suspicious ({path_diversity:.1f}%)")
    
    print(f"\nResult: {checks_passed}/{total_checks} checks passed")
    return checks_passed == total_checks

In [ ]:
# Run sanity checks on all models
df = data['SPY']
train_prices = df['close'].to_numpy()[:-FORECAST_STEPS]
test_prices = df['close'].to_numpy()[-FORECAST_STEPS:]

sanity_results = {}

for name, result in comparison_forecasts.items():
    passed = run_sanity_checks(result, train_prices, test_prices, name)
    sanity_results[name] = passed

print("\n" + "="*60)
print("SANITY CHECK SUMMARY")
print("="*60)
for name, passed in sanity_results.items():
    status = "✓ PASS" if passed else "✗ FAIL"
    print(f"{name}: {status}")

## 8. Summary

Final summary and recommendations.

In [ ]:
print("\n" + "="*80)
print("CLOUDCONDITIONEDLSTM EVALUATION SUMMARY")
print("="*80)

# Best configuration
if self_comparison_results:
    best_self = min(self_comparison_results, key=lambda x: x['mape'])
    print(f"\nBest Self-Configuration: {best_self['model']}")
    print(f"  MAPE: {best_self['mape']:.1f}%")
    print(f"  Direction Accuracy: {best_self['direction_accuracy']:.1f}%")
    print(f"  Coverage: {best_self['coverage']:.1f}%")

# Comparison ranking
if comparison_results:
    print(f"\nModel Ranking (by MAPE):")
    sorted_results = sorted(comparison_results, key=lambda x: x['mape'])
    for i, r in enumerate(sorted_results, 1):
        print(f"  {i}. {r['model']}: MAPE={r['mape']:.1f}%, Dir={r['direction_accuracy']:.1f}%")

# Multi-asset performance
if multi_asset_results:
    print(f"\nMulti-Asset Performance (CloudConditionedLSTM):")
    avg_mape = np.mean([r['mape'] for r in multi_asset_results])
    avg_dir = np.mean([r['direction_accuracy'] for r in multi_asset_results])
    avg_coverage = np.mean([r['coverage'] for r in multi_asset_results])
    print(f"  Average MAPE: {avg_mape:.1f}%")
    print(f"  Average Direction Accuracy: {avg_dir:.1f}%")
    print(f"  Average Coverage: {avg_coverage:.1f}%")

print("\n" + "="*80)

In [ ]:
# Recommendations
print("\nRECOMMENDATIONS:")
print("-" * 40)

if comparison_results:
    cloud_result = next((r for r in comparison_results if r['model'] == 'CloudConditionedLSTM'), None)
    if cloud_result:
        rank = sorted_results.index(cloud_result) + 1
        
        if rank == 1:
            print("✓ CloudConditionedLSTM is the best performing model!")
        elif rank <= 2:
            print(f"✓ CloudConditionedLSTM ranks #{rank} - competitive performance")
        else:
            print(f"⚠ CloudConditionedLSTM ranks #{rank} - may need tuning")

# Suggest best cloud_weight based on self-comparison
if self_comparison_results:
    for result in self_comparison_results:
        if 'HighCloud' in result['model']:
            high_cloud_mape = result['mape']
        elif 'LowCloud' in result['model']:
            low_cloud_mape = result['mape']
        elif 'Default' in result['model']:
            default_mape = result['mape']
    
    try:
        if high_cloud_mape < default_mape and high_cloud_mape < low_cloud_mape:
            print("→ Consider higher cloud_weight (0.5-0.7) for this asset")
        elif low_cloud_mape < default_mape and low_cloud_mape < high_cloud_mape:
            print("→ Consider lower cloud_weight (0.1-0.2) for this asset")
        else:
            print("→ Default cloud_weight (0.3) is working well")
    except:
        pass

print("\n✓ Evaluation complete!")